# Merbok 2022 hourly Hs animation
Instantaneous WW3/CMEPS `wavImp_Sw_Hs` hourly animation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image
import cartopy.crs as ccrs
import cartopy.feature as cfeature

INDIR = Path('/glade/derecho/scratch/hyeonmin/alaska_ocn_wav_ice.002/run')
FILE_GLOB = 'alaska_ocn_wav_ice.002.cpl.hx.ww3.*.nc'
VAR = 'wavImp_Sw_Hs'
LONVAR, LATVAR = 'wavImp_lon', 'wavImp_lat'
START, END = '2022-09-14 00:00', '2022-09-20 23:59'
OUTDIR = Path('/glade/u/home/hyeonmin/analysis/merbok_analysis')
OUTDIR.mkdir(parents=True, exist_ok=True)
GIF_OUT = OUTDIR / 'merbok_hourly_Hs.gif'
files = sorted(INDIR.glob(FILE_GLOB))
print('files:', len(files))
if not files: raise FileNotFoundError(INDIR / FILE_GLOB)

In [ ]:
parts=[]
for f in files:
    with xr.open_dataset(f, decode_times=True) as d:
        if VAR not in d: raise KeyError(f'{VAR} not found in {f.name}')
        parts.append(d[[VAR,LONVAR,LATVAR]].load())
ds=xr.concat(parts,dim='time').sortby('time')
_,idx=np.unique(ds.time.values,return_index=True)
ds=ds.isel(time=np.sort(idx)).sel(time=slice(START,END))
print('records:',ds.sizes['time'],pd.Timestamp(ds.time.values[0]),'to',pd.Timestamp(ds.time.values[-1]))

In [ ]:
lon_da,lat_da=ds[LONVAR],ds[LATVAR]
if 'time' in lon_da.dims: lon_da=lon_da.isel(time=0)
if 'time' in lat_da.dims: lat_da=lat_da.isel(time=0)
lon=((np.asarray(lon_da,dtype=float)+180)%360)-180
lat=np.asarray(lat_da,dtype=float)
hs=np.asarray(ds[VAR].values,dtype=float)
hs[(~np.isfinite(hs))|(np.abs(hs)>=1e20)]=np.nan
hs=np.where(hs>0,hs,np.nan)
pos=hs[np.isfinite(hs)]
if not pos.size: raise RuntimeError('No positive Hs found')
vmax=max(1.0,float(np.ceil(np.nanpercentile(pos,99.5))))
print('color scale: 0 -',vmax,'m')

In [ ]:
fig=plt.figure(figsize=(9,6.5))
ax=plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(resolution='50m',linewidth=.8)
ax.add_feature(cfeature.LAND,facecolor='silver',zorder=2)
gl=ax.gridlines(draw_labels=True,linewidth=.4,alpha=.5)
gl.top_labels=False; gl.right_labels=False
mesh=ax.pcolormesh(lon,lat,hs[0],shading='auto',vmin=0,vmax=vmax,transform=ccrs.PlateCarree())
cb=fig.colorbar(mesh,ax=ax,pad=.03,shrink=.85); cb.set_label('Significant wave height, Hs (m)')
title=ax.set_title('')
def update(i):
    mesh.set_array(np.ma.masked_invalid(hs[i]).ravel())
    t=pd.Timestamp(ds.time.values[i]); a=hs[i]
    title.set_text(f'Merbok 2022 — NO ICE | {t:%Y-%m-%d %H:%M} UTC\nHs max={np.nanmax(a):.2f} m | mean={np.nanmean(a):.2f} m')
    return mesh,title
ani=FuncAnimation(fig,update,frames=len(ds.time),interval=180,blit=False)
ani.save(GIF_OUT,writer=PillowWriter(fps=6),dpi=130)
plt.close(fig)
print('Saved:',GIF_OUT)

In [ ]:
display(Image(filename=str(GIF_OUT)))